# NB22: Temporal Political Models

This notebook demonstrates the **temporal political modeling** layer in siege_utilities.
These models map political activity to time periods and geographic units:

- **CongressionalTerm** — the atomic 2-year unit of political time
- **Seat** — a political identity that persists across redistricting (e.g., "U.S. House CA-12")
- **StateElectionCalendar** — per-state, per-cycle primary/general dates
- **Race** — a contest for a Seat in a Term (regular or special election)
- **RaceEvent** — milestones in a race timeline (filing, primary, general, certification)
- **SpatioTemporalEvent** — real-world events with geometry (hurricanes, court rulings)
- **ReturnSnapshot** — progressive election-night result reporting

All models have corresponding **Pydantic schemas** for validation without a database.

## 1. Pydantic Schemas (No Database Required)

The schemas mirror the Django models but work standalone — great for validation,
API serialization, and data pipeline intermediate representations.

In [ ]:
from datetime import date, datetime

from siege_utilities.geo.schemas.temporal_political import (
    CongressionalTermSchema,
    SeatSchema,
    StateElectionCalendarSchema,
)
from siege_utilities.geo.schemas.temporal_events import (
    RaceSchema,
    RaceEventSchema,
    SpatioTemporalEventSchema,
    ReturnSnapshotSchema,
)

### 1.1 CongressionalTerm — The Atomic Unit of Political Time

In [ ]:
# The 119th Congress (2025-2027), elected in November 2024
term_119 = CongressionalTermSchema(
    congress_number=119,
    start_date=date(2025, 1, 3),
    end_date=date(2027, 1, 3),
    election_year=2024,
    is_presidential=True,
    senate_classes_up=[2],  # Class II Senators were up in 2024
)
print(f"Congress: {term_119.congress_number}")
print(f"Election year: {term_119.election_year}")
print(f"Presidential cycle: {term_119.is_presidential}")
print(f"Senate classes up: {term_119.senate_classes_up}")
print(f"Term: {term_119.start_date} to {term_119.end_date}")

In [ ]:
# Build a sequence of recent Congresses
recent_terms = []
for n, year in [(116, 2018), (117, 2020), (118, 2022), (119, 2024)]:
    term = CongressionalTermSchema(
        congress_number=n,
        start_date=date(year + 1, 1, 3),
        end_date=date(year + 3, 1, 3),
        election_year=year,
        is_presidential=(year % 4 == 0),
    )
    recent_terms.append(term)
    print(f"  {n}th Congress: {year} election, presidential={term.is_presidential}")

### 1.2 Seat — Political Identity Across Redistricting

In [ ]:
# A seat persists even when district boundaries change
house_seat = SeatSchema(
    office="US_HOUSE",
    state_fips="06",      # California
    district_label="12",   # CA-12
)
print(f"House seat: {house_seat.office} {house_seat.state_fips}-{house_seat.district_label}")

senate_seat = SeatSchema(
    office="US_SENATE",
    state_fips="48",      # Texas
    senate_class=2,
)
print(f"Senate seat: {senate_seat.office} {senate_seat.state_fips} Class {senate_seat.senate_class}")

president = SeatSchema(
    office="PRESIDENT",
    # No state — it's national
)
print(f"President: {president.office} (state={president.state_fips})")

### 1.3 StateElectionCalendar — Per-State Election Dates

In [ ]:
# California's 2024 election calendar
ca_calendar = StateElectionCalendarSchema(
    state_fips="06",
    congress_number=119,
    primary_date=date(2024, 3, 5),
    primary_type="top_two",
    general_date=date(2024, 11, 5),
    registration_deadline=date(2024, 10, 21),
    early_voting_start=date(2024, 10, 7),
    early_voting_end=date(2024, 11, 4),
    mail_ballot_request_deadline=date(2024, 10, 29),
    certification_deadline=date(2024, 12, 11),
)
print(f"State: {ca_calendar.state_fips}")
print(f"Primary: {ca_calendar.primary_date} ({ca_calendar.primary_type})")
print(f"General: {ca_calendar.general_date}")
print(f"Registration deadline: {ca_calendar.registration_deadline}")
print(f"Early voting: {ca_calendar.early_voting_start} to {ca_calendar.early_voting_end}")
print(f"Certification: {ca_calendar.certification_deadline}")

### 1.4 Race — Contests for Seats

In [ ]:
# Regular race
regular = RaceSchema(seat_id=1, congress_number=119)
print(f"Regular race: special={regular.is_special}")

# Special election (e.g., after a resignation)
special = RaceSchema(
    seat_id=42,
    congress_number=118,
    is_special=True,
    cause_of_special="resignation",
)
print(f"Special election: cause={special.cause_of_special}")

### 1.5 SpatioTemporalEvent — Real-World Events with Geometry

In [ ]:
# A court ruling that affected redistricting
rucho = SpatioTemporalEventSchema(
    feature_id="rucho-v-common-cause-2019",
    name="Rucho v. Common Cause",
    vintage_year=2019,
    event_start=datetime(2019, 6, 27, 10, 0),
    event_category="COURT_RULING",
    description="Supreme Court ruled partisan gerrymandering claims are non-justiciable",
)
print(f"Event: {rucho.name}")
print(f"Category: {rucho.event_category}")
print(f"Date: {rucho.event_start}")
print(f"Duration: {'ongoing' if rucho.event_end is None else rucho.event_end}")

# A natural disaster that disrupted an election
helene = SpatioTemporalEventSchema(
    feature_id="hurricane-helene-2024",
    name="Hurricane Helene",
    vintage_year=2024,
    event_start=datetime(2024, 9, 24),
    event_end=datetime(2024, 9, 28),
    event_category="NATURAL_DISASTER",
    description="Category 4 hurricane affecting NC/GA/FL, disrupting voter access",
)
print(f"\nEvent: {helene.name}")
print(f"Duration: {helene.event_start.date()} to {helene.event_end.date()}")

### 1.6 RaceEvent — Milestones in a Race Timeline

In [ ]:
# Build a race timeline
timeline = [
    RaceEventSchema(race_id=1, event_type="FILING_OPEN", event_date=date(2023, 11, 1)),
    RaceEventSchema(race_id=1, event_type="FILING_CLOSE", event_date=date(2024, 1, 15)),
    RaceEventSchema(race_id=1, event_type="PRIMARY", event_date=date(2024, 3, 5),
                    event_start=datetime(2024, 3, 5, 7, 0),
                    event_end=datetime(2024, 3, 5, 20, 0)),
    RaceEventSchema(race_id=1, event_type="GENERAL", event_date=date(2024, 11, 5),
                    event_start=datetime(2024, 11, 5, 6, 0),
                    event_end=datetime(2024, 11, 5, 20, 0)),
    RaceEventSchema(race_id=1, event_type="CERTIFICATION", event_date=date(2024, 12, 11)),
]

print("Race Timeline:")
for evt in timeline:
    polls = ""
    if evt.event_start and evt.event_end:
        polls = f" (polls {evt.event_start.strftime('%I%p')}-{evt.event_end.strftime('%I%p')})"
    print(f"  {evt.event_date}  {evt.event_type}{polls}")

### 1.7 ReturnSnapshot — Progressive Election Night Results

In [ ]:
# Simulate progressive election night reporting
snapshots = [
    ReturnSnapshotSchema(
        race_id=1,
        timestamp=datetime(2024, 11, 5, 20, 30),
        source="ap",
        precincts_reporting=120,
        total_precincts=2400,
        pct_reporting=5.0,
        results={
            "smith_d": {"votes": 8500, "pct": 54.2, "party": "DEM"},
            "jones_r": {"votes": 7200, "pct": 45.8, "party": "REP"},
        },
    ),
    ReturnSnapshotSchema(
        race_id=1,
        timestamp=datetime(2024, 11, 5, 23, 0),
        source="ap",
        precincts_reporting=1800,
        total_precincts=2400,
        pct_reporting=75.0,
        results={
            "smith_d": {"votes": 125000, "pct": 52.1, "party": "DEM"},
            "jones_r": {"votes": 115000, "pct": 47.9, "party": "REP"},
        },
    ),
    ReturnSnapshotSchema(
        race_id=1,
        timestamp=datetime(2024, 12, 10, 14, 0),
        source="state_sos",
        pct_reporting=100.0,
        is_final=True,
        results={
            "smith_d": {"votes": 168000, "pct": 51.8, "party": "DEM"},
            "jones_r": {"votes": 156000, "pct": 48.2, "party": "REP"},
        },
    ),
]

print("Election Night Reporting:")
for snap in snapshots:
    status = "FINAL" if snap.is_final else f"{snap.pct_reporting:.0f}% in"
    leader = max(snap.results.items(), key=lambda x: x[1]["votes"])
    print(f"  {snap.timestamp.strftime('%H:%M')} [{status}] {snap.source.upper()}: "
          f"{leader[0]} leads with {leader[1]['pct']:.1f}%")

## 2. Validation

Pydantic enforces constraints automatically — no bad data gets through.

In [ ]:
from pydantic import ValidationError

# Congress number must be 1-200
try:
    CongressionalTermSchema(
        congress_number=0,  # Invalid!
        start_date=date(2025, 1, 3),
        end_date=date(2027, 1, 3),
        election_year=2024,
    )
except ValidationError as e:
    print(f"Caught validation error: {e.errors()[0]['msg']}")

# Senate class must be 1-3
try:
    SeatSchema(office="US_SENATE", state_fips="06", senate_class=4)
except ValidationError as e:
    print(f"Caught validation error: {e.errors()[0]['msg']}")

# pct_reporting must be 0-100
try:
    ReturnSnapshotSchema(
        race_id=1,
        timestamp=datetime.now(),
        source="ap",
        pct_reporting=101.0,  # Invalid!
    )
except ValidationError as e:
    print(f"Caught validation error: {e.errors()[0]['msg']}")

print("\nAll validations working correctly.")

## 3. Seed Data Generator

The `populate_congressional_terms` management command generates historical data
for all Congresses. Here we use the underlying function directly.

In [ ]:
# The populate_congressional_terms command uses these helpers.
# We demonstrate the logic directly (no Django setup needed).

def senate_classes_up(election_year: int) -> list:
    """Which Senate classes are up in a given election year?"""
    mod = election_year % 6
    if mod == 0: return [3]
    elif mod == 2: return [1]
    elif mod == 4: return [2]
    return []

# Generate info for recent + upcoming Congresses
print("Congressional Terms (110th-125th):")
for n in range(110, 126):
    election_year = 1788 + (n - 1) * 2
    start_year = 1935 + (n - 74) * 2
    is_pres = (election_year % 4 == 0)
    classes = senate_classes_up(election_year)
    print(f"  {n:>3}th: election={election_year}, start={start_year}, "
          f"presidential={is_pres}, senate_classes={classes}")

print("\nSenate class rotation (2018-2030):")
for year in range(2018, 2031, 2):
    classes = senate_classes_up(year)
    print(f"  {year}: Class {classes} up for election")


## Summary

| Model | Purpose | Key Fields |
|-------|---------|------------|
| **CongressionalTerm** | 2-year time unit | congress_number, election_year, is_presidential |
| **Seat** | Political identity | office, state, district_label, senate_class |
| **StateElectionCalendar** | Per-state dates | primary_date, general_date, registration_deadline |
| **Race** | Contest for a seat | seat, term, is_special, cause_of_special |
| **RaceEvent** | Timeline milestones | event_type, event_date, external_event |
| **SpatioTemporalEvent** | Real-world events | geometry, event_category, affected_boundaries |
| **ReturnSnapshot** | Result reporting | results JSON, pct_reporting, is_final |

**Design principles:**
- Seat is identity, not geography (persists across redistricting)
- CongressionalTerm is the atomic time unit
- Events own their own geometry (independent of boundaries)
- ReturnSnapshot enables progressive result tracking